In [4]:
import bibtexparser
import yaml
import re

# Mapping BibTeX ENTRYTYPE to your defined YAML type
TYPE_MAP = {
    "article": "journal",
    "book": "book",
    "inbook": "book",
    "incollection": "book",
    "inproceedings": "paper",
    "conference": "paper",
    "techreport": "paper",
    "manual": "paper",
    "misc": "preprint",
    "unpublished": "unpublished",
    "mastersthesis": "unpublished",
    "phdthesis": "unpublished",
}

# Valid prefixes allowed by Manubot and your rules
ALLOWED_PREFIXES = {"doi", "pmid", "pmc", "arxiv", "isbn", "wikidata", "url", "handle"}

def extract_arxiv_id(url):
    match = re.search(r'arxiv\.org/(?:abs|pdf)/([0-9]+\.[0-9]+)', url)
    return match.group(1) if match else None

def get_prefixed_id(entry):
    if 'doi' in entry:
        return f"doi:{entry['doi'].strip()}"
    elif 'pmid' in entry:
        return f"pmid:{entry['pmid'].strip()}"
    elif 'pmc' in entry:
        return f"pmc:{entry['pmc'].strip()}"
    elif 'isbn' in entry:
        return f"isbn:{entry['isbn'].strip()}"
    elif 'wikidata' in entry:
        return f"wikidata:{entry['wikidata'].strip()}"
    elif 'handle' in entry:
        return f"handle:{entry['handle'].strip()}"
    elif 'url' in entry:
        url = entry['url'].strip()
        if 'arxiv.org' in url:
            arxiv_id = extract_arxiv_id(url)
            if arxiv_id:
                return f"arxiv:{arxiv_id}"
        return f"url:{url}"
    return None  # No valid ID found

def bibtex_to_yaml(bibtex_path, yaml_path):
    with open(bibtex_path, "r", encoding="utf-8") as bibfile:
        bib_database = bibtexparser.load(bibfile)

    entries_out = []
    for entry in bib_database.entries:
        prefixed_id = get_prefixed_id(entry)
        if not prefixed_id:
            continue  # skip entries without valid ID

        yaml_entry = {"id": prefixed_id}

        # Type mapping
        entry_type = entry.get("ENTRYTYPE", "").lower()
        yaml_entry["type"] = TYPE_MAP.get(entry_type, "paper")

        # Common fields
        if "title" in entry:
            yaml_entry["title"] = entry["title"]
        if "author" in entry:
            yaml_entry["authors"] = [a.strip() for a in entry["author"].replace("\n", " ").split(" and ")]
        if "journal" in entry:
            yaml_entry["container-title"] = entry["journal"]
        if "year" in entry:
            yaml_entry["date"] = entry["year"]
        if "publisher" in entry:
            yaml_entry["publisher"] = entry["publisher"]
        if "volume" in entry:
            yaml_entry["volume"] = entry["volume"]
        if "number" in entry:
            yaml_entry["issue"] = entry["number"]
        if "pages" in entry:
            yaml_entry["pages"] = entry["pages"]
        if "doi" in entry:
            yaml_entry["doi"] = entry["doi"]
        if "url" in entry:
            yaml_entry["link"] = entry["url"]

        entries_out.append(yaml_entry)

    # Write to YAML
    with open(yaml_path, "w", encoding="utf-8") as outfile:
        yaml.dump(entries_out, outfile, sort_keys=False, allow_unicode=True)

    print(f"✅ Wrote {len(entries_out)} valid entries to {yaml_path}")

# --- Run the conversion ---
bibtex_to_yaml("input.bib", "output.yaml")


✅ Wrote 0 valid entries to output.yaml
